In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine
import os

def connect_to_db():
    engine = create_engine(
        f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
        f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    )
    return engine

In [ ]:
engine = connect_to_db()

conn = engine.connect()

In [ ]:
df = pd.read_sql("SELECT * FROM public.gold_model_dataset", engine)

df.head()

In [ ]:
aggregated_features = [
    "user_booking_rate",
    "avg_session_intensity",
    "avg_stay_duration",
    "avg_advance_booking_days",
    "mobile_usage_rate",
    "destination_booking_rate",
    "destination_popularity",
    "cluster_booking_rate",
    "cluster_popularity"
]

In [ ]:
df[aggregated_features].hist(figsize=(14, 10))
plt.tight_layout()

In [ ]:
def plot_feature_vs_target(feature, bins=10):
    temp = df[[feature, "is_booking"]].copy()
    temp["bin"] = pd.qcut(temp[feature], q=bins, duplicates="drop")

    grouped = temp.groupby("bin")["is_booking"].mean()

    grouped.plot(kind="bar", figsize=(10,4))
    plt.title(f"{feature} vs Booking Rate")
    plt.ylabel("Booking Rate")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
plot_feature_vs_target("user_booking_rate")
plot_feature_vs_target("destination_booking_rate")
plot_feature_vs_target("cluster_booking_rate")

In [ ]:
corr = df[aggregated_features + ["is_booking"]].corr()

corr["is_booking"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
df[["cluster_booking_rate", "is_booking"]].head()

In [ ]:
df["user_high"] = df["user_booking_rate"] > df["user_booking_rate"].median()

df.groupby("user_high")["is_booking"].mean()

In [ ]:
conn.close()

## Key Insights

- Aggregated features such as `user_booking_rate` and `destination_booking_rate` show a strong relationship with the target variable, indicating high predictive potential.
- Users with higher historical booking rates are significantly more likely to convert, reinforcing the importance of behavioral history.
- Destination-level metrics also contribute meaningful signal, suggesting that certain destinations inherently drive higher conversion.
- Features derived from user behavior and context (e.g., session intensity, booking window) demonstrate moderate but consistent relationships with booking outcomes.
- Some features exhibit extremely high correlation with the target, indicating potential data leakage and requiring careful interpretation.

## Conclusion

The feature engineering process successfully generated variables with strong predictive power. While several features provide meaningful signal, special attention must be given to those derived directly from the target (e.g., aggregated booking rates), as they may introduce leakage. Overall, the dataset is well-prepared for modeling, with a solid mix of behavioral, contextual, and aggregated features.